# Capstone · Rural Hospital Closures
*Intro to Python for Scientists & Public Health Professionals*

This running project threads through three lessons. You'll take a messy dataset of rural hospital closures, clean it, summarize it, and enrich it with a second table:

- **Part 1 — Cleaning** (this lesson)
- **Part 2 — GroupBy** (Lesson 8)
- **Part 3 — Merge** (Lesson 10)

**About the data.** This is a *teaching dataset* modeled on the real **UNC Cecil G. Sheps Center** rural hospital closures tracker — same kinds of fields (state, rurality, Medicare payment type, closure year, beds, services remaining, complete vs. converted closure). The values here are constructed for practice and include deliberately planted data-quality problems. The real, authoritative tracker (≈197 closures since 2005, kept current) lives here:
<https://www.shepscenter.unc.edu/programs-projects/rural-health/rural-hospital-closures/>

### How this notebook works
Each task is a prompt with an empty cell to try, and a collapsed **Solution** you can expand. Work Part 1 now; Parts 2 and 3 are staged for later lessons.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"
df = pd.read_csv(f"{BASE_URL}/rural_hospital_closures.csv")
df.head()

# Part 1 — Cleaning

Goal: turn this raw load into an analysis-ready table.

### Task 1 — First look
What is the shape, and what do the columns and dtypes look like?

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
print(df.shape)
df.info()
```

**Why.** Always orient yourself first: how many rows and columns, what types came in, and where the obvious gaps are. `info()` shows non-null counts, which previews the missing-data work ahead.

</details>

### Task 2 — Drop the throwaway index column
The file was saved with a row number that pandas reads back as `Unnamed: 0`. Remove it.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
df = df.drop(columns="Unnamed: 0")
df.columns.tolist()
```

**Why.** That column is just a saved row counter — it carries no information and would only get in the way. (You could also have read the file with `index_col=0`.)

</details>

### Task 3 — Fix the miscoded closure year
One row has a `closure_year` of `1019` — a typo for `2019`. Correct it.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
df["closure_year"] = df["closure_year"].replace(1019, 2019)
df["closure_year"].min(), df["closure_year"].max()
```

**Why.** A single bad value distorts any time analysis (and min/max sanity checks). `replace` swaps just that value, leaving the rest untouched.

</details>

### Task 4 — Standardize `closure_type`
It should contain only `Complete` and `Converted`, but the raw values include mixed case and misspellings. Clean them up.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
df["closure_type"] = df["closure_type"].str.strip().str.capitalize()
df["closure_type"] = df["closure_type"].replace({"Complet": "Complete", "Convertd": "Converted"})
df["closure_type"].value_counts()
```

**Why.** `.str.capitalize()` fixes the case issues (`CONVERTED` -> `Converted`) in one vectorized pass, but it can't repair true misspellings — so a small `replace` map handles `Complet` and `Convertd`. Inconsistent categories silently split your later group counts, so standardizing now prevents wrong totals.

</details>

### Task 5 — Fix a data type
`hospital_id` is an identifier, not a quantity. Store it as text.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
df["hospital_id"] = df["hospital_id"].astype("string")
df["hospital_id"].dtype
```

**Why.** IDs look numeric but you'd never average them. Making it text prevents accidental arithmetic and keeps it safe as a key later.

</details>

### Task 6 — Remove duplicate rows
There are exactly three fully-duplicated rows. Drop them.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
print("duplicates:", df.duplicated().sum())
df = df.drop_duplicates()
print("after:", df.duplicated().sum())
```

**Why.** Duplicate records double-count in every summary. `drop_duplicates` keeps the first of each group by default.

</details>

### Task 7 — Inspect missing data
Which columns have missing values, and how much? A quick chart helps.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print(missing_pct[missing_pct > 0])

ax = missing_pct[missing_pct > 0].plot.barh()
ax.set_xlabel("% missing"); plt.tight_layout(); plt.show()
```

**Why.** Seeing the *amount* missing per column drives the next decisions: a little missingness gets imputed, a lot gets dropped.

</details>

### Task 8 — Impute missing bed counts
`beds` has a handful of missing values. Fill them with the column's median.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
df["beds"] = df["beds"].fillna(df["beds"].median())
df["beds"].isna().sum()
```

**Why.** The median is robust to the extreme bed counts still lurking in the data (we remove those in Task 12), so it's a safer fill than the mean here.

</details>

### Task 9 — Drop columns that are mostly empty
Drop any column missing more than 35% of its values.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
to_drop = df.columns[df.isna().mean() > 0.35]
print("dropping:", list(to_drop))
df = df.drop(columns=to_drop)
```

**Why.** `reopened_year` is empty for ~94% of rows (most closed hospitals never reopen). Imputing a column that's mostly missing would invent data; dropping it is the honest move. Writing it as a rule (`> 0.35`) makes it reusable.

</details>

### Task 10 — Drop rows missing a state
A few rows have no `state`. Without it they can't be grouped or mapped, so drop them.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
before = len(df)
df = df.dropna(subset=["state"])
print("dropped", before - len(df), "rows")
```

**Why.** `dropna(subset=[...])` removes rows missing a value in just that column. Use this sparingly — only when the missing field is essential and can't be recovered.

</details>

### Task 11 — Quick EDA
Make one categorical chart (e.g., closures by `closure_type` or `payment_type`) and one histogram of `beds`.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
sns.countplot(x="closure_type", data=df); plt.title("Closures by type"); plt.show()
sns.histplot(data=df, x="beds", binwidth=10); plt.title("Bed counts"); plt.show()
```

**Why.** A glance at the distributions confirms the cleaning worked (two clean categories) and reveals the bed-count outliers you'll handle next.

</details>

### Task 12 — Remove bed-count outliers
Rural hospitals are small; a few bed counts are clearly erroneous. Remove outliers using the 1.5 × IQR rule, then report the final shape.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
q1, q3 = df["beds"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
before = len(df)
df = df[df["beds"].between(low, high)]
print("removed", before - len(df), "outlier rows")
print("final shape:", df.shape)
```

**Why.** The 1.5 × IQR rule is the same one a boxplot's whiskers use — a defensible, non-arbitrary cutoff. The 600- and 720-bed entries fall far outside it and are removed.

</details>

> **Save your cleaned `df`** — Parts 2 and 3 build on it.

# Part 2 — GroupBy *(Lesson 8)*

Using your cleaned `df`, work these after the GroupBy lesson:

1. How many closures are there per `state`? Which state has the most?
2. What is the mean number of `beds` for `Complete` vs. `Converted` closures?
3. How many closures of each `payment_type` occurred, and what is their mean bed count?
4. Count closures per `closure_year` to see the trend over time.

In [ ]:
# Part 2 — your work here


<details>
<summary>Solution</summary>

```python
# 1. closures per state (ranked)
print(df.groupby("state").size().sort_values(ascending=False).head())

# 2. mean beds by closure type
print(df.groupby("closure_type")["beds"].mean())

# 3. count and mean beds by payment type
print(df.groupby("payment_type").agg(n=("hospital_id", "size"),
                                      avg_beds=("beds", "mean")))

# 4. closures per year
print(df.groupby("closure_year").size())
```

**Why this works.** `groupby(...).size()` counts rows per group; sorting ranks them. Named `.agg` returns the count and mean side by side with clean column names. Grouping by `closure_year` produces a count per year — a time trend that sets up the datetime lesson coming next.

</details>

# Part 3 — Merge *(Lesson 10)*

The `payment_type` column holds codes (`CAH`, `PPS`, …). A lookup table maps each code to its full description:

`https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main/payment_types.csv`

1. Read the lookup table.
2. Merge it onto your cleaned `df` so each row gains a readable `description`.
3. Check: are the join keys the same dtype? What kind of join keeps every closure even if a code is missing from the lookup?

In [ ]:
# Part 3 — your work here (Lesson 10)


<details>
<summary>Solution</summary>

```python
lookup = pd.read_csv(f"{BASE_URL}/payment_types.csv")

# 1 & 3: check the join keys are the same dtype (both text here)
print(df["payment_type"].dtype, lookup["code"].dtype)

# 2: a LEFT join keeps every closure, even if a code were missing from the lookup
merged = df.merge(lookup, left_on="payment_type", right_on="code", how="left")
merged[["hospital", "payment_type", "description"]].head()
```

**Why this works.** The closures table holds payment-type *codes*; the lookup maps each code to a full *description*. Merging `left_on="payment_type"` to `right_on="code"` brings the description alongside each closure. A **left** join is the safe default for adding a lookup — it never drops your main rows, and any code missing from the lookup simply gets `NaN`. Both keys are text, so they compare correctly (a string-vs-integer mismatch would silently match nothing).

</details>